In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, filters, measure, color
import cv2
from pathlib import Path


In [ ]:
#Helperfunction for plotting

def show_image(image, title="", cmap="gray"):
    plt.figure(figsize=(5,5))
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()

def show_results_grid(images, titles, cols=3, figsize=(15,10), cmap="gray"):
    rows = int(np.ceil(len(images) / cols))
    fig, axs = plt.subplots(rows, cols, figsize=figsize)
    axs = axs.ravel()
    for i in range(len(images)):
        axs[i].imshow(images[i], cmap=cmap if images[i].ndim==2 else None)
        axs[i].set_title(titles[i])
        axs[i].axis("off")
    for i in range(len(images), len(axs)):
        axs[i].axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
#Load Images and Preprocess

def load_image(path):
    image = io.imread(path)
    return color.rgb2gray(image) if image.ndim==3 else image

def preprocess_image(image):
    return cv2.GaussianBlur(image, (5,5), 0)


In [ ]:
def threshold_image(image, method="otsu"):
    methods = {
        "otsu": filters.threshold_otsu,
        "li": filters.threshold_li,
        "triangle": filters.threshold_triangle
    }
    thresh = methods[method.lower()](image)
    return image > thresh

def region_growing(binary_image):
    _, labels = cv2.connectedComponents(binary_image.astype("uint8"))
    return labels

def analyze_regions(labels, min_fraction=0.1):
    regions = measure.regionprops(labels)
    areas = [r.area for r in regions]
    avg_area = np.mean(areas) if areas else 0
    min_area = avg_area * min_fraction
    filtered = np.zeros_like(labels)
    for r in regions:
        if r.area >= min_area:
            filtered[labels==r.label] = r.label
    return filtered

def create_bitmask(filtered_labels):
    return (filtered_labels > 0).astype(np.uint8)


In [ ]:
def overlay_red_channel(bitmask, rgb_image):
    overlay = rgb_image.copy()
    if overlay.ndim == 2:  # grayscale → RGB
        overlay = np.stack((overlay,)*3, axis=-1)
    overlay[...,0] = np.clip(overlay[...,0] + bitmask*255, 0, 255)
    return overlay


In [ ]:
def compute_mask_stats(bitmask, gray_image):
    region_pixels = gray_image[bitmask>0]
    stats = {
        "area": int((bitmask>0).sum()),
        "mean_intensity": float(region_pixels.mean()) if len(region_pixels)>0 else 0,
        "std_intensity": float(region_pixels.std()) if len(region_pixels)>0 else 0,
        "min_intensity": float(region_pixels.min()) if len(region_pixels)>0 else 0,
        "max_intensity": float(region_pixels.max()) if len(region_pixels)>0 else 0
    }
    return stats

In [ ]:
def process_thresholds(image_gray, rgb_image, methods=["otsu","li","triangle"]):
    results = []
    for method in methods:
        binary = threshold_image(image_gray, method)
        labels = region_growing(binary)
        filtered = analyze_regions(labels)
        bitmask = create_bitmask(filtered)
        overlay = overlay_red_channel(bitmask, rgb_image)
        stats = compute_mask_stats(bitmask, image_gray)
        results.append({
            "method": method,
            "bitmask": bitmask,
            "overlay": overlay,
            "stats": stats
        })
    return results


In [ ]:
root = "data"
blue_imgs = list(Path(root).rglob("*Blue.tif"))

all_results = []

for blue_path in blue_imgs:
    print(f"Processing: {blue_path.name}")

    gray_img = load_image(blue_path)
    gray_preprocessed = preprocess_image(gray_img)

    # Find corresponding RGB image (replace "Blue" with "Red")
    rgb_path = str(blue_path).replace("Blue", "Red")
    rgb_img = io.imread(rgb_path)

    results = process_thresholds(gray_preprocessed, rgb_img)

    for r in results:
        r["filename"] = blue_path.name  # add filename to result

    all_results.extend(results)

